In [ ]:
import csv
from pathlib import Path

import torch
from PIL import Image
from torchvision import transforms
from torchvision.models import SqueezeNet1_1_Weights, squeezenet1_1

# Folder gambar dan lokasi file hasil
image_dir = Path(
    r"D:\00 Kuliah\TA\Experiment\Dataset\Data Gambar\gambar_terpilih"
)
output_csv = Path(
    r"D:\00 Kuliah\TA\Experiment\Model\5 ML\Ekstraksi_fitur_squeezenet.csv"
)

batch_size = 8
expected_features = 169_000
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

if not image_dir.is_dir():
    raise NotADirectoryError(f"Folder gambar tidak ditemukan: {image_dir}")

image_files = sorted(
    path
    for path in image_dir.iterdir()
    if path.is_file() and path.suffix.lower() in image_extensions
)
if not image_files:
    raise FileNotFoundError(f"Tidak ada gambar yang didukung di: {image_dir}")

# Pilih GPU jika tersedia; jika tidak, gunakan CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = SqueezeNet1_1_Weights.DEFAULT
model = squeezenet1_1(weights=weights).to(device)
model.eval()

# Ukuran input mengikuti catatan metode: 227 x 227.
# Mean dan std tetap mengikuti normalisasi ImageNet bobot pretrained.
weight_preprocess = weights.transforms()
preprocess = transforms.Compose(
    [
        transforms.Resize(227),
        transforms.CenterCrop((227, 227)),
        transforms.ToTensor(),
        transforms.Normalize(mean=weight_preprocess.mean, std=weight_preprocess.std),
    ]
)

output_csv.parent.mkdir(parents=True, exist_ok=True)
feature_names = [f"fitur_{index:06d}" for index in range(1, expected_features + 1)]
processed = 0

# Tulis per batch agar seluruh matriks fitur tidak perlu disimpan sekaligus di RAM.
with output_csv.open("w", newline="", encoding="utf-8-sig") as output_file:
    writer = csv.writer(output_file)
    writer.writerow(["nama_file", *feature_names])

    for start in range(0, len(image_files), batch_size):
        batch_paths = image_files[start : start + batch_size]
        batch_images = []

        for image_path in batch_paths:
            with Image.open(image_path) as image:
                batch_images.append(preprocess(image.convert("RGB")))

        batch_tensor = torch.stack(batch_images).to(device)

        with torch.inference_mode():
            # Conv1 dan Fire2-Fire9 berada di model.features.
            feature_maps = model.features(batch_tensor)
            # classifier[1] adalah Conv10; classifier[2] adalah ReLU.
            conv10_maps = model.classifier[1](feature_maps)
            relu_conv10 = model.classifier[2](conv10_maps)

            if tuple(relu_conv10.shape[1:]) != (1000, 13, 13):
                raise RuntimeError(
                    "Bentuk keluaran relu_conv10 tidak sesuai metode: "
                    f"diharapkan (1000, 13, 13), didapat {tuple(relu_conv10.shape[1:])}"
                )

            feature_vectors = relu_conv10.flatten(start_dim=1).cpu().numpy()

        if feature_vectors.shape[1] != expected_features:
            raise RuntimeError(
                f"Diharapkan {expected_features} fitur per gambar, "
                f"tetapi didapat {feature_vectors.shape[1]}"
            )

        for image_path, vector in zip(batch_paths, feature_vectors, strict=True):
            writer.writerow([image_path.name, *vector.tolist()])

        processed += len(batch_paths)
        print(f"Diproses: {processed}/{len(image_files)} gambar")

print(f"Jumlah gambar diproses: {processed}")
print(f"Dimensi fitur per gambar: {expected_features}")
print(f"Bentuk feature map relu_conv10: {tuple(relu_conv10.shape[1:])}")
print(f"File hasil: {output_csv}")